In [1]:
# Cell 1 - Setup
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/ML_final')

!pip install -q prophet

import pandas as pd
import numpy as np

Mounted at /content/drive


In [2]:
# Cell 2 - Verify everything imports cleanly
from prophet import Prophet
print('Prophet imported OK')

Prophet imported OK


In [3]:
# Cell 3 - Load processed data
train = pd.read_pickle('data/train_processed.pkl')
train = train.sort_values(['Store','Dept','Date'])
print(train['Date'].min(), train['Date'].max())

2010-02-05 00:00:00 2012-10-26 00:00:00


In [4]:
# Cell 4 - WMAE metric
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

In [5]:
# Cell 5 - Prophet has a built-in mechanism for holiday effects - pass it
# the exact holiday dates already identified in feature engineering
# instead of relying on Prophet's generic US-holidays calendar, since
# Walmart's four flagged holidays are what the competition actually scores
# 5x weight on.
superbowl = ['2010-02-12','2011-02-11','2012-02-10','2013-02-08']
laborday  = ['2010-09-10','2011-09-09','2012-09-07','2013-09-06']
thanksgiving = ['2010-11-26','2011-11-25','2012-11-23','2013-11-29']
christmas = ['2010-12-31','2011-12-30','2012-12-28','2013-12-27']

holidays_df = pd.concat([
    pd.DataFrame({'holiday': 'SuperBowl', 'ds': pd.to_datetime(superbowl)}),
    pd.DataFrame({'holiday': 'LaborDay', 'ds': pd.to_datetime(laborday)}),
    pd.DataFrame({'holiday': 'Thanksgiving', 'ds': pd.to_datetime(thanksgiving)}),
    pd.DataFrame({'holiday': 'Christmas', 'ds': pd.to_datetime(christmas)}),
])
holidays_df['lower_window'] = -1
holidays_df['upper_window'] = 1
holidays_df.head()

,holiday,ds,lower_window,upper_window
0,SuperBowl,2010-02-12,-1,1
1,SuperBowl,2011-02-11,-1,1
2,SuperBowl,2012-02-10,-1,1
3,SuperBowl,2013-02-08,-1,1
0,LaborDay,2010-09-10,-1,1


In [12]:
# Cell 6 - Prophet is also a LOCAL model - same sampling rationale as
# ARIMA/SARIMA.
SAMPLE_SIZE = 15
VAL_LEN = 39
MIN_LEN = 52 + VAL_LEN + 10

series_totals = (
    train.groupby(['Store','Dept'])['Weekly_Sales']
    .sum()
    .sort_values(ascending=False)
)
np.random.seed(42)
all_keys = series_totals.index.tolist()
sample_keys = list(np.random.choice(len(all_keys), size=SAMPLE_SIZE, replace=False))
sample_keys = [all_keys[i] for i in sample_keys]

series_data = {}
for store, dept in sample_keys:
    sub = train[(train['Store']==store) & (train['Dept']==dept)].sort_values('Date')
    if len(sub) < MIN_LEN:
        continue
    sub = sub.set_index('Date').asfreq('W-FRI')
    sub['Weekly_Sales'] = sub['Weekly_Sales'].interpolate().bfill().ffill()
    sub['IsHoliday'] = sub['IsHoliday'].astype(float).fillna(0.0).astype(bool)
    series_data[(store, dept)] = sub.reset_index()[['Date','Weekly_Sales','IsHoliday']]
    if len(series_data) >= SAMPLE_SIZE:
        break

print('Series selected for Prophet:', len(series_data))

Series selected for Prophet: 13


In [13]:
# Cell 7 - Same 39-week holdout used everywhere else
fit_df, val_df = {}, {}
for key, df in series_data.items():
    fit_df[key] = df.iloc[:-VAL_LEN].copy()
    val_df[key] = df.iloc[-VAL_LEN:].copy()

In [14]:
# Cell 8 - Fit Prophet per series. Prophet needs columns named exactly
# 'ds' and 'y'.
import logging
logging.getLogger('prophet').setLevel(logging.ERROR)
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)

prophet_models = {}
all_true, all_pred, all_holiday = [], [], []

for key, df in fit_df.items():
    prophet_train = df.rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})[['ds', 'y']]

    model = Prophet(
        holidays=holidays_df,
        yearly_seasonality=True,
        weekly_seasonality=False,   # data is already weekly-aggregated
        daily_seasonality=False,
        seasonality_mode='additive'
    )
    model.fit(prophet_train)
    prophet_models[key] = model

    future = model.make_future_dataframe(periods=VAL_LEN, freq='W-FRI', include_history=False)
    forecast = model.predict(future)

    all_true.extend(val_df[key]['Weekly_Sales'].values)
    all_pred.extend(forecast['yhat'].values)
    all_holiday.extend(val_df[key]['IsHoliday'].values)

    print(f"{key}: fit done")

all_true = np.array(all_true)
all_pred = np.array(all_pred)
all_holiday = np.array(all_holiday)
print('Done. Total validation points:', len(all_true))

(41, 13): fit done
(26, 28): fit done
(32, 67): fit done
(28, 30): fit done
(18, 27): fit done
(15, 32): fit done
(44, 92): fit done
(18, 36): fit done
(15, 6): fit done
(8, 29): fit done
(16, 52): fit done
(16, 79): fit done
(24, 14): fit done
Done. Total validation points: 507


In [15]:
# Cell 9 - Evaluate
prophet_wmae = wmae(all_true, all_pred, all_holiday)
print('Prophet WMAE:', prophet_wmae)
print('Holiday WMAE:    ', wmae(all_true[all_holiday], all_pred[all_holiday], all_holiday[all_holiday]))
print('Non-holiday WMAE:', wmae(all_true[~all_holiday], all_pred[~all_holiday], all_holiday[~all_holiday]))

Prophet WMAE: 925.1282458112571
Holiday WMAE:     844.7263454881708
Non-holiday WMAE: 946.8584891418208


In [ ]:
# Cell 10 - Save models
import os
import joblib

os.makedirs('models', exist_ok=True)
joblib.dump(prophet_models, 'models/prophet_best.pkl')
print('Saved.')